# Training VD-HMM Models with CmdStanPy

This notebook is dedicated to training **VD-HMM** models for S = 2, 3, and 4.

**Note:**

To train standard **HMM** models, please refer to notebook 2b.
If you are reproducing results using notebook cells, make sure to run both this notebook and notebook 2b concurrently.

If you want to run training outside of notebook run:

```bash
  uv run scripts/main.py --model vdhmm --seed {seed} --state {number of states} --chains {number of chains} --parallel-chains {number of parallel chains}

```

Look at the `main.py` file in the `scripts` folder for full documentation


In [1]:
import sys
from pathlib import Path

# Add project root to path
sys.path.append(str(Path("..").resolve()))

import pickle
import numpy as np
from helpers import ModelData
import cmdstanpy

SEED = 42  # set seed to train with correct indices (change, if desired)
SET_ORIGINAL_INDICES = False  # if set to true, the original paper indices are selected

print(f"CmdStanPy Version: {cmdstanpy.__version__}")
print(f"CmdStan Path: {cmdstanpy.cmdstan_path()}")

CmdStanPy Version: 1.3.0
CmdStan Path: /Users/omidsedighi-mornani/.cmdstan/cmdstan-2.37.0


/Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load processed data (with seed)
from constants import PROCESSED_DATA_FOLDER


if SET_ORIGINAL_INDICES:
    data_path = PROCESSED_DATA_FOLDER / f"processed_data_{SEED}.pkl"
else:
    data_path = PROCESSED_DATA_FOLDER / f"processed_data_original.pkl"

model_data = ModelData.from_pickle(data_path)

print(model_data.summary())


ModelData Summary:
States (HMM): Not set
Total businesses: 921
Training samples: 500
Total observations: 64887
Number of covariates: 16

Data shapes:
- Ratings: 64887
- Sentiment: 64887
- Days: 64887
- Q matrix: (500, 16)
- R matrix: (16, 16)
- X_test: (421, 16)

Business status:
- Closed: 225
- Open: 696

Preprocessing artifacts: Available
- Train indices: 500
- Calibration indices: 100
- Eval indices: 321

Benchmark data: Not available



In [3]:
from constants import FITTED_MODEL_FOLDER, STAN_MODEL_FOLDER
from helpers import prepare_stan_data


def train_model_cmdstan(
    model_data,
    S,
    model_name="vdhmm",
    chains=2,
    parallel_chains=2,
    iter_warmup=500,
    iter_sampling=500,
    seed=42,
    adapt_delta=0.8,
    max_treedepth=10,
):
    """
    Trainiert ein Modell mit CmdStanPy.

    Parameters:
    -----------
    model_data : ModelData
        Daten für das Training
    S : int
        Anzahl der Hidden States (2-5)
    model_name : str
        'vdhmm' oder 'hmm'
    chains : int
        Anzahl der MCMC Chains
    parallel_chains : int
        Anzahl parallel laufender Chains (nutzt parallel_chains CPU Cores)
    iter_warmup : int
        Warmup Iterationen
    iter_sampling : int
        Sampling Iterationen (post-warmup)
    seed : int
        Random Seed
    adapt_delta : float
        Stan adapt_delta Parameter (0.8-0.99, höher = konservativer)
    max_treedepth : int
        Stan max_treedepth Parameter

    Returns:
    --------
    cmdstanpy.CmdStanMCMC : Fit-Objekt
    """
    assert model_name in ["vdhmm", "hmm"], f"Invalid model_name: {model_name}"
    assert S in range(2, 6), "S must be between 2 and 5"

    # Daten vorbereiten
    stan_data = prepare_stan_data(model_data, S)

    # Model file
    model_file = STAN_MODEL_FOLDER / f"{model_name}.stan"
    if not model_file.exists():
        raise FileNotFoundError(f"Stan model not found: {model_file}")

    print(f"\n{'='*60}")
    print(f"Training {model_name.upper()} with S={S} states (CmdStanPy)")
    print(f"{'='*60}")
    print(f"Model file: {model_file}")
    print(f"\nConfiguration:")
    print(f"  Chains: {chains}")
    print(f"  Parallel chains: {parallel_chains}")
    print(f"  Warmup iterations: {iter_warmup}")
    print(f"  Sampling iterations: {iter_sampling}")
    print(f"  Total iterations: {iter_warmup + iter_sampling}")
    print(f"  Seed: {seed}")
    print(f"  Adapt delta: {adapt_delta}")
    print(f"  Max treedepth: {max_treedepth}")

    # Compile model
    print(f"\nCompiling model...")
    model = cmdstanpy.CmdStanModel(stan_file=str(model_file))
    print(f"✓ Model compiled")

    # Sample
    print(f"\nSampling...")
    fit = model.sample(
        data=stan_data,
        chains=chains,
        parallel_chains=parallel_chains,
        iter_warmup=iter_warmup,
        iter_sampling=iter_sampling,
        seed=seed,
        adapt_delta=adapt_delta,
        max_treedepth=max_treedepth,
        show_progress=True,
        inits=0,
    )

    print(f"\n✓ Sampling complete!")

    # Save model
    output_path = FITTED_MODEL_FOLDER / f"{model_name}_{S}_cmdstan.pkl"
    with open(output_path, "wb") as f:
        pickle.dump(
            {
                "fit": fit,
                "model_name": model_name,
                "S": S,
                "stan_data": stan_data,
                "summary": fit.summary(),
            },
            f,
        )

    print(f"✓ Model saved to {output_path}")

    # Diagnostics
    print(f"\n{'-'*60}")
    print("Diagnostics:")
    print(f"{'-'*60}")
    print(fit.diagnose())

    # Summary statistics
    print(f"\n{'-'*60}")
    print("Summary (first 20 parameters):")
    print(f"{'-'*60}")
    summary_df = fit.summary()
    print(summary_df.head(20))

    return fit

## Training Configuration

**Paper-Standard:** 2 chains × 1000 iterations (500 warmup + 500 sampling)

**Für schnelles Testen:** 2 chains × 200 iterations (100 warmup + 100 sampling)


In [4]:
# Training settings
SEED = 42
CHAINS = 4
PARALLEL_CHAINS = 4  # Uses 4 CPU cores in parallel
ITER_WARMUP = 1000  # Same as in the original paper
ITER_SAMPLING = 1000  # Same as in the original paper
ADAPT_DELTA = 0.95  # Range: 0.8-0.95, increase if there are divergent transitions
MAX_TREEDEPTH = 10  # Typical range: 10-15

np.random.seed(SEED)

print("Training Configuration:")
print(f"  Seed: {SEED}")
print(f"  Chains: {CHAINS}")
print(f"  Parallel chains: {PARALLEL_CHAINS}")
print(f"  Warmup iterations: {ITER_WARMUP}")
print(f"  Sampling iterations: {ITER_SAMPLING}")
print(f"  Total iterations: {ITER_WARMUP + ITER_SAMPLING}")
print(f"  Total posterior samples: {CHAINS * ITER_SAMPLING}")

Training Configuration:
  Seed: 42
  Chains: 4
  Parallel chains: 4
  Warmup iterations: 1000
  Sampling iterations: 1000
  Total iterations: 2000
  Total posterior samples: 4000


## Training VD-HMM Models (S=2, 3, 4)

Variable-Duration Hidden Markov Models mit zeitabhängigen Übergangswahrscheinlichkeiten.


In [5]:
# Dictionary zum Speichern aller Modelle
trained_models = {}

# VD-HMM Training für S=2 bis S=4
for S in range(2, 4 + 1):
    try:
        print(f"\n\n{'#'*60}")
        print(f"# VD-HMM Training: S={S}")
        print(f"{'#'*60}\n")

        fit = train_model_cmdstan(
            model_data=model_data,
            S=S,
            model_name="vdhmm",
            chains=CHAINS,
            parallel_chains=PARALLEL_CHAINS,
            iter_warmup=ITER_WARMUP,
            iter_sampling=ITER_SAMPLING,
            seed=SEED,
            adapt_delta=ADAPT_DELTA,
            max_treedepth=MAX_TREEDEPTH,
        )

        trained_models[f"vdhmm_{S}"] = fit
        print(f"\n✓✓✓ VD-HMM with S={S} completed successfully! ✓✓✓\n")

    except Exception as e:
        print(f"\n✗✗✗ Error training VD-HMM with S={S}: {e} ✗✗✗\n")
        raise

print("\n" + "=" * 60)
print("VD-HMM Training Complete!")
print("=" * 60)



############################################################
# VD-HMM Training: S=2
############################################################


Training VDHMM with S=2 states (CmdStanPy)
Model file: /Users/omidsedighi-mornani/Desktop/STUDIUM WIRTSCHAFTSINFORMATIK/Semester 5/Seminar Gust/SourceCode/data/stan_code/vdhmm.stan

Configuration:
  Chains: 4
  Parallel chains: 4
  Warmup iterations: 1000
  Sampling iterations: 1000
  Total iterations: 2000
  Seed: 42
  Adapt delta: 0.95
  Max treedepth: 10

Compiling model...
✓ Model compiled

Sampling...


15:50:17 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/2000 [00:00<?, ?it/s, (Warmup)]


chain 1:   0%|          | 1/2000 [00:00<23:29,  1.42it/s, (Warmup)]


15:50:37 - cmdstanpy - ERROR - Chain [1] error: code '-2' Unknown error: -2
15:50:37 - cmdstanpy - ERROR - Chain [2] error: code '-2' Unknown error: -2
15:50:37 - cmdstanpy - ERROR - Chain [3] error: code '-2' Unknown error: -2
15:50:37 - cmdstanpy - ERROR - Chain [4] error: code '-2' Unknown error: -2


KeyboardInterrupt: 

## Training Summary


In [ ]:
# Summary
print("\n" + "=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"\nTotal models trained: {len(trained_models)}")
print(f"Models: {list(trained_models.keys())}")
print(f"\nSaved in: {FITTED_MODEL_FOLDER}")

# List saved models
saved_models = sorted(FITTED_MODEL_FOLDER.glob("vdhmm_*_cmdstan.pkl"))
print(f"\nSaved VD-HMM model files ({len(saved_models)}):")
for model_file in saved_models:
    size_mb = model_file.stat().st_size / (1024 * 1024)
    print(f"  - {model_file.name} ({size_mb:.2f} MB)")